In [2]:
import os
import copy
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import albumentations as A
import torch.nn.functional as F

import cv2
import torch
import torch.nn as nn

from tqdm import tqdm
from torch.utils.data import DataLoader
from PIL import Image
from torch.utils.tensorboard import SummaryWriter
from sklearn.decomposition import PCA

import my_utils


from sklearn.metrics import precision_recall_curve, auc
import PATT_UNet

from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor

import segmentation_models_pytorch as smp

In [3]:
label_path = r'core_dataset\masks'
srez_path = r'core_dataset\images'

In [4]:
def cross_datasets(datasets_prefix):
    train_files = []
    val_files = []
    for dataset in datasets_prefix:
        dataset_files = [
            f for f in os.listdir(label_path)
            if f.startswith(dataset)
        ]


        train_files.extend(dataset_files[:128])
        val_files.extend(dataset_files[128:168])


    train_image = []
    train_target = []

    for fname in train_files:
        label = cv2.imread(os.path.join(label_path, fname))
        srez = cv2.imread(os.path.join(srez_path, fname))

        train_image.append(srez[:, :, 0])
        train_target.append(label[:, :, 0])


    val_image = []
    val_target = []

    for fname in val_files:
        label = cv2.imread(os.path.join(label_path, fname))
        srez = cv2.imread(os.path.join(srez_path, fname))

        val_image.append(srez[:, :, 0])
        val_target.append(label[:, :, 0])

    return {"train_image": train_image,
            "train_target": train_target,
            "val_image": val_image,
            "val_target": val_target,
            }

In [5]:
req_size = 512
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=30, p=0.5),
    A.RandomRotate90(p=0.5),

    # A.GaussianBlur(p = 0.25, sigma_limit = [0.5, 1]),
    # A.GaussNoise(p = 0.5),
    # A.Downscale(scale_range=[0.5, 0.75], p = 0.25),

    # A.RandomResizedCrop(size = [256, 256], scale = [0.8, 1.0]),
    A.Resize(req_size, req_size),
    A.Normalize(),
], additional_targets={'target': 'mask'})
val_transform = A.Compose([
    A.Resize(req_size, req_size),
    A.Normalize(),
], additional_targets={'target': 'mask'})

In [6]:
# model = PATT_UNet.PAttUNet(input_channels = 1, num_classes = 1)
# model = my_utils.MAEWithDecoder(torch.load(r'weights\MAE ViT\last.pth', weights_only=False))
model = smp.Segformer(in_channels = 1, classes = 1)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total:,}")
print(f"Trainable params: {trainable:,}")

Total params: 21,870,273
Trainable params: 21,870,273


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 8
num_epochs = 30
learning_rate = 2e-4
warmup_epochs = 10

def lr_lambda(current_epoch):
    if current_epoch < warmup_epochs:
        return float(current_epoch + 1) / warmup_epochs
    return 1.0 

optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
    )

warmup_scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lr_lambda
)

plateau_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',         
    factor=0.2,    
    patience=3,            
    min_lr=1e-8,
    threshold = 1e-3          
)

In [8]:
torch.save(model , r'weights/Segformer/no.pth')

In [9]:
model.to(device)

log_dir = f"Cross-Validation/SegFormer {datetime.now().strftime('%d_%m %H_%M')}"
writer = SummaryWriter(log_dir=log_dir)

loss_fn = nn.BCEWithLogitsLoss()
dice_loss = my_utils.DiceLoss()

datasets = ['Beton',
            'data3d',
            'DRP421Bentheimer',
            'DRP421Leopard',
            'DRP58',
            ]



train_data = cross_datasets(datasets)

train_dataset = my_utils.CoreDataset(train_data['train_target'], train_data['train_image'], transform, multiply_channels = False)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

total_batches = len(train_loader) * num_epochs
with tqdm(total=total_batches, desc='Training', unit='batch') as pbar:

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        for batch in train_loader:
            images = batch["image"].to(device).unsqueeze(1)   # [B, С, H, W]
            targets = batch["target"].to(device) # [B, H, W]

            pred = model(images)

            loss = loss_fn(pred, targets.unsqueeze(1)) + dice_loss(pred, targets)

            train_loss += loss.item() / batch_size

            loss.backward()

            optimizer.step()
            optimizer.zero_grad()

            pbar.update(1)

        train_loss /= len(train_loader)

        if epoch < warmup_epochs:
            warmup_scheduler.step()
        else:
            plateau_scheduler.step(train_loss)

        writer.add_scalar(f'Loss/Train', train_loss, epoch)

torch.save(model , r'weights/Segformer/all.pth')

Training:  67%|██████▋   | 4305/6400 [22:15<10:50,  3.22batch/s]


KeyboardInterrupt: 